# Variance Ratio Demo

This notebook checks the variance gain from aggregating a whole nonzero frequency orbit.

For an $n$-qubit exponential encoding circuit with $L$ upload layers, the scalar frequency spectrum is
$$
\Omega=\llbracket-\frac{3^{nL}-1}{2},\frac{3^{nL}-1}{2}\rrbracket.
$$

We use two orbit frequencies:
$$
0 \longleftrightarrow \{0\},\qquad
\nu_n \longleftrightarrow \Omega\setminus\{0\}.
$$

The experiment uses orbit frequency $\nu_n$. Thus
$$
|[\nu_n]|=|\Omega|-1=3^{nL}-1,
$$
and the aggregated coefficient is
$$
a_{\nu_n}=\sum_{\omega\in\Omega\setminus\{0\}}c_\omega.
$$

The plotted ratio is
$$
\rho_n=
\frac{\operatorname{Var}_\theta(a_{\nu_n})}
{\overline{\operatorname{Var}_\theta(c_\omega)}_{\omega\in\Omega\setminus\{0\}}}.
$$
Under ideal 2-design decoupling, $\rho_n$ should be close to $|[\nu_n]|$.


In [132]:
from dataclasses import dataclass
from pathlib import Path
from typing import Optional
import itertools
import math
import time

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import torch


def paper_work_dir():
    cwd = Path.cwd()
    for candidate in (cwd, *cwd.parents):
        if candidate.name == "Paper_work":
            return candidate
        if (candidate / "Paper_work").is_dir():
            return candidate / "Paper_work"
    return cwd


PAPER_WORK = paper_work_dir()
FIGURE_PATH = PAPER_WORK / "MSQE" / "figures" / "variance_ratio.png"
RESULTS_PATH = PAPER_WORK / "MSQE" / "figures" / "variance_ratio_results.npz"

# Change these top-level knobs; both B=n and B=1 settings below use them.
QUBIT_COUNTS = (1, 2, 3, 4, 5)
N_UPLOAD_LAYERS = 2
ENCODING_BASE = 3
N_PARAMETER_SAMPLES = 96
MAX_GRID_POINTS = 1_000_000


def binomial(n, k):
    if hasattr(math, "comb"):
        return math.comb(n, k)
    return math.factorial(n) // (math.factorial(k) * math.factorial(n - k))


@dataclass
class ExperimentConfig:
    qubit_counts: tuple = QUBIT_COUNTS
    n_upload_layers: int = N_UPLOAD_LAYERS
    encoding_base: int = ENCODING_BASE
    n_parameter_samples: int = N_PARAMETER_SAMPLES
    blocks_per_trainable_layer: object = None
    setting_name: str = "deeper setting"
    seed: int = 20260521
    prefer_mps: bool = True
    max_grid_points: int = MAX_GRID_POINTS
    real_dtype: torch.dtype = torch.float32
    complex_dtype: torch.dtype = torch.complex64

    @property
    def omega_max(self):
        return sum(self.encoding_base**layer for layer in range(self.n_upload_layers))

    @property
    def nx(self):
        # Nyquist grid for integer frequencies in [-omega_max, omega_max].
        return 2 * self.omega_max + 1


CONFIG = ExperimentConfig(
    qubit_counts=QUBIT_COUNTS,
    n_upload_layers=N_UPLOAD_LAYERS,
    encoding_base=ENCODING_BASE,
    n_parameter_samples=N_PARAMETER_SAMPLES,
    max_grid_points=MAX_GRID_POINTS,
    setting_name=r"StrongEntangle blocks, $B=n$",
)
BLOCK1_CONFIG = ExperimentConfig(
    qubit_counts=CONFIG.qubit_counts,
    n_upload_layers=CONFIG.n_upload_layers,
    encoding_base=CONFIG.encoding_base,
    n_parameter_samples=CONFIG.n_parameter_samples,
    blocks_per_trainable_layer=1,
    setting_name=r"StrongEntangle blocks, $B=1$",
    seed=CONFIG.seed + 2718,
    prefer_mps=CONFIG.prefer_mps,
    max_grid_points=CONFIG.max_grid_points,
    real_dtype=CONFIG.real_dtype,
    complex_dtype=CONFIG.complex_dtype,
)
mpl.rcParams.update({
    "font.size": 15,
    "axes.labelsize": 17,
    "axes.titlesize": 18,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 13,
    "axes.linewidth": 1.15,
    "figure.dpi": 140,
})

In [ ]:
def complex_exp(angle):
    """Return exp(i * angle) without relying on Python complex scalar promotion."""
    return torch.complex(torch.cos(angle), torch.sin(angle))


def choose_torch_device(prefer_mps=True):
    candidates = []
    if prefer_mps and torch.backends.mps.is_available():
        candidates.append("mps")
    if torch.cuda.is_available():
        candidates.append("cuda")
    candidates.append("cpu")

    for name in candidates:
        device = torch.device(name)
        try:
            test = torch.eye(2, dtype=torch.complex64, device=device)
            batch = torch.ones((4, 2), dtype=torch.complex64, device=device)
            index = torch.tensor([1, 0], dtype=torch.long, device=device)
            _ = (test @ test).real.sum().item()
            _ = torch.matmul(batch, test.T).real.sum().item()
            _ = torch.index_select(batch, -1, index).real.sum().item()
            return device
        except Exception as exc:
            print(f"Skipping {name}: complex matmul is unavailable ({exc})")
    return torch.device("cpu")


DEVICE = choose_torch_device(CONFIG.prefer_mps)
print(f"Using torch device: {DEVICE}")
print(f"grid size per n: Nx = {CONFIG.nx}, omega_max = {CONFIG.omega_max}")

## Batched Statevector Simulator

The original notebook evaluated one PennyLane QNode for every grid point and every parameter sample. Here the whole input grid is propagated as a batch of statevectors. This is much faster for the small-to-medium systems used in this coefficient variance check, and it can use `mps` if the local PyTorch build supports the required complex operations.

In [134]:
def basis_bits(n_qubits, device):
    # Precompute bit tables on CPU. This avoids MPS integer bitwise-kernel limitations.
    dim = 2**n_qubits
    bits = np.array(
        [[(basis >> (n_qubits - 1 - wire)) & 1 for wire in range(n_qubits)] for basis in range(dim)],
        dtype=np.int64,
    )
    return torch.tensor(bits, dtype=torch.long, device=device)


def input_grid(n_qubits, nx, device, dtype):
    values = torch.linspace(0.0, 2.0 * math.pi, nx + 1, dtype=dtype)[:-1]
    points = torch.cartesian_prod(*([values] * n_qubits))
    if points.ndim == 1:
        points = points[:, None]
    return points.to(device)


def initial_states(n_grid, n_qubits, device, complex_dtype):
    states = torch.zeros((n_grid, 2**n_qubits), dtype=complex_dtype, device=device)
    states[:, 0] = 1.0 + 0.0j
    return states


def rz_gate(theta, complex_dtype):
    diagonal = torch.stack([complex_exp(-0.5 * theta), complex_exp(0.5 * theta)]).to(complex_dtype)
    return torch.diag(diagonal)


def ry_gate(theta, complex_dtype):
    c = torch.cos(0.5 * theta)
    s = torch.sin(0.5 * theta)
    return torch.stack([
        torch.stack([c, -s]),
        torch.stack([s, c]),
    ]).to(complex_dtype)


def rot_gate(phi, theta, omega, complex_dtype):
    # PennyLane Rot(phi, theta, omega) = RZ(omega) RY(theta) RZ(phi).
    return rz_gate(omega, complex_dtype) @ ry_gate(theta, complex_dtype) @ rz_gate(phi, complex_dtype)


def apply_single_qubit_gate(states, gate, wire, n_qubits):
    original_shape = states.shape
    tensor = states.reshape(-1, *([2] * n_qubits))
    tensor = torch.movedim(tensor, wire + 1, -1)
    tensor = torch.matmul(tensor, gate.T)
    tensor = torch.movedim(tensor, -1, wire + 1)
    return tensor.reshape(original_shape)


def cnot_gather_index(n_qubits, control, target, device):
    dim = 2**n_qubits
    gather = np.empty(dim, dtype=np.int64)
    for old_index in range(dim):
        bits = [(old_index >> (n_qubits - 1 - wire)) & 1 for wire in range(n_qubits)]
        new_bits = bits.copy()
        if bits[control] == 1:
            new_bits[target] ^= 1
        new_index = 0
        for bit in new_bits:
            new_index = (new_index << 1) | bit
        gather[new_index] = old_index
    return torch.tensor(gather, dtype=torch.long, device=device)


def entangler_pairs(n_qubits, block_index):
    if n_qubits <= 1:
        return []
    entangling_range = (block_index % (n_qubits - 1)) + 1
    return [(wire, (wire + entangling_range) % n_qubits) for wire in range(n_qubits)]


def apply_trainable_layer(states, weights_layer, cnot_cache, n_qubits, complex_dtype):
    n_blocks = weights_layer.shape[0]
    for block in range(n_blocks):
        for wire in range(n_qubits):
            phi, theta, omega = weights_layer[block, wire]
            gate = rot_gate(phi, theta, omega, complex_dtype)
            states = apply_single_qubit_gate(states, gate, wire, n_qubits)
        for control, target in entangler_pairs(n_qubits, block):
            states = torch.index_select(states, -1, cnot_cache[(control, target)])
    return states


def encoding_signs(n_qubits, device, dtype):
    bits = basis_bits(n_qubits, device).to(dtype)
    return bits - 0.5


def apply_data_encoding(states, grid, signs, beta):
    # RZ(beta * x_j) on wire j gives phase exp(i * beta * x_j * (bit_j - 1/2)).
    phase_angles = (beta * grid) @ signs.T
    return states * complex_exp(phase_angles).to(states.dtype)


def local_projector_observable_diag(n_qubits, device, dtype):
    # O = (1/n) sum_j |0><0|_j is diagonal; value is the fraction of zero bits.
    bits = basis_bits(n_qubits, device).to(dtype)
    return 1.0 - bits.mean(dim=1)


def observable_expectation(states, observable_diag):
    probabilities = states.abs().square()
    return probabilities @ observable_diag

In [ ]:
def trainable_block_count(n_qubits, config):
    rule = config.blocks_per_trainable_layer
    if rule is None or rule == "n":
        return max(int(n_qubits), 1)
    if callable(rule):
        return max(int(rule(n_qubits)), 1)
    return max(int(rule), 1)


def make_cache(n_qubits, config, device):
    n_grid = config.nx**n_qubits
    if n_grid > config.max_grid_points:
        raise ValueError(
            f"Grid has {n_grid:,} points for n={n_qubits}. "
            f"Reduce qubit_counts, n_upload_layers, or raise max_grid_points."
        )

    n_blocks = trainable_block_count(n_qubits, config)
    cnot_cache = {
        pair: cnot_gather_index(n_qubits, pair[0], pair[1], device)
        for block in range(n_blocks)
        for pair in entangler_pairs(n_qubits, block)
    }
    return {
        "grid": input_grid(n_qubits, config.nx, device, config.real_dtype),
        "signs": encoding_signs(n_qubits, device, config.real_dtype),
        "observable_diag": local_projector_observable_diag(n_qubits, device, config.real_dtype),
        "cnot_cache": cnot_cache,
    }


def random_weights(n_qubits, config, generator, device):
    n_blocks = trainable_block_count(n_qubits, config)
    shape = (config.n_upload_layers + 1, n_blocks, n_qubits, 3)
    weights = 2.0 * math.pi * torch.rand(shape, generator=generator, dtype=config.real_dtype)
    return weights.to(device)


def evaluate_function_grid(n_qubits, weights, cache, config, device):
    grid = cache["grid"]
    states = initial_states(grid.shape[0], n_qubits, device, config.complex_dtype)

    states = apply_trainable_layer(
        states, weights[0], cache["cnot_cache"], n_qubits, config.complex_dtype
    )
    for layer in range(config.n_upload_layers):
        beta = config.encoding_base**layer
        states = apply_data_encoding(states, grid, cache["signs"], beta)
        states = apply_trainable_layer(
            states, weights[layer + 1], cache["cnot_cache"], n_qubits, config.complex_dtype
        )

    values = observable_expectation(states, cache["observable_diag"])
    return values.reshape((config.nx,) * n_qubits).detach().cpu().numpy()


def coefficient_tensor(function_grid):
    # If f(x)=sum_omega c_omega exp(-i omega.x), ifftn gives c_omega on the integer grid.
    return np.fft.ifftn(function_grid)


def coefficient_at(coeffs, omega):
    idx = tuple(int(w) % coeffs.shape[axis] for axis, w in enumerate(omega))
    return coeffs[idx]

## PennyLane Equivalence Check

This diagnostic checks that the custom batched Torch simulator implements the same circuit as PennyLane's `StronglyEntanglingLayers` for the settings used below. It compares expectation values for the same inputs and the same random trainable weights.

In [135]:
def evaluate_custom_points(n_qubits, x_points, weights, config, device):
    x_points = np.asarray(x_points, dtype=float)
    if x_points.ndim == 1:
        x_points = x_points[None, :]
    grid = torch.tensor(x_points, dtype=config.real_dtype, device=device)

    n_blocks = trainable_block_count(n_qubits, config)
    cnot_cache = {
        pair: cnot_gather_index(n_qubits, pair[0], pair[1], device)
        for block in range(n_blocks)
        for pair in entangler_pairs(n_qubits, block)
    }
    signs = encoding_signs(n_qubits, device, config.real_dtype)
    observable_diag = local_projector_observable_diag(n_qubits, device, config.real_dtype)

    states = initial_states(grid.shape[0], n_qubits, device, config.complex_dtype)
    states = apply_trainable_layer(states, weights[0], cnot_cache, n_qubits, config.complex_dtype)
    for layer in range(config.n_upload_layers):
        beta = config.encoding_base**layer
        states = apply_data_encoding(states, grid, signs, beta)
        states = apply_trainable_layer(states, weights[layer + 1], cnot_cache, n_qubits, config.complex_dtype)

    return observable_expectation(states, observable_diag).detach().cpu().numpy()


def pennylane_point_values(n_qubits, x_points, weights, config):
    import pennylane as qml

    x_points = np.asarray(x_points, dtype=float)
    if x_points.ndim == 1:
        x_points = x_points[None, :]
    weights_np = weights.detach().cpu().numpy()
    observable_diag = local_projector_observable_diag(n_qubits, torch.device("cpu"), torch.float64).numpy()
    observable_matrix = np.diag(observable_diag)
    wires = list(range(n_qubits))
    strongly_entangling = qml.templates.StronglyEntanglingLayers

    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev)
    def circuit(x_vec, trainable_weights):
        strongly_entangling(trainable_weights[0], wires=wires)
        for layer in range(config.n_upload_layers):
            beta = config.encoding_base**layer
            for wire in wires:
                qml.RZ(beta * x_vec[wire], wires=wire)
            strongly_entangling(trainable_weights[layer + 1], wires=wires)
        return qml.expval(qml.Hermitian(observable_matrix, wires=wires))

    return np.array([float(circuit(x_vec, weights_np)) for x_vec in x_points], dtype=float)


def verify_pennylane_equivalence(tolerance=5e-5):
    n_qubits = 3
    x_points = np.array(
        [
            [0.10, 0.70, 1.30],
            [1.20, 2.10, 0.40],
            [2.70, 0.30, 3.00],
            [5.10, 4.20, 1.70],
        ],
        dtype=float,
    )
    settings = [
        (r"B=n", CONFIG),
        (r"B=1", BLOCK1_CONFIG),
    ]

    for label, config in settings:
        generator = torch.Generator(device="cpu")
        generator.manual_seed(config.seed + 12345)
        weights = random_weights(n_qubits, config, generator, DEVICE)
        custom_values = evaluate_custom_points(n_qubits, x_points, weights, config, DEVICE)
        pennylane_values = pennylane_point_values(n_qubits, x_points, weights, config)
        abs_error = np.abs(custom_values - pennylane_values)
        print(f"{label}: max |custom - PennyLane| = {abs_error.max():.3e}")
        print(np.column_stack([custom_values, pennylane_values, abs_error]))
        assert abs_error.max() < tolerance, f"{label} mismatch: {abs_error.max():.3e}"


verify_pennylane_equivalence()


B=n: max |custom - PennyLane| = 3.940e-07
[[1.61743224e-01 1.61743304e-01 8.02279707e-08]
 [5.99167347e-01 5.99167397e-01 4.98161021e-08]
 [7.94582665e-01 7.94582357e-01 3.07832271e-07]
 [6.46416008e-01 6.46416402e-01 3.93953159e-07]]
B=1: max |custom - PennyLane| = 1.380e-07
[[3.19904357e-01 3.19904399e-01 4.17476675e-08]
 [4.67001170e-01 4.67001281e-01 1.11299785e-07]
 [4.03946221e-01 4.03946359e-01 1.38046795e-07]
 [3.70267123e-01 3.70267242e-01 1.19405147e-07]]


## Orbit Statistics

For each $n$ and $L$, the frequency spectrum is
$$
\Omega=\llbracket-\frac{3^{nL}-1}{2},\frac{3^{nL}-1}{2}\rrbracket.
$$

The orbit frequencies used in this notebook are
$$
0 \longleftrightarrow \{0\},\qquad
\nu_n \longleftrightarrow \Omega\setminus\{0\}.
$$

The variance experiment uses $\nu_n$, so
$$
|[\nu_n]|=3^{nL}-1,
\qquad
a_{\nu_n}=\sum_{\omega\in\Omega\setminus\{0\}}c_\omega.
$$

The denominator of the ratio is estimated by averaging the empirical variances of the individual $c_\omega$ over $\omega\in\Omega\setminus\{0\}$.


In [136]:
def upload_betas(config):
    return tuple(int(config.encoding_base**layer) for layer in range(config.n_upload_layers))


def coordinate_frequency_values(config):
    values = {0}
    for beta in upload_betas(config):
        values = {freq + sign * beta for freq in values for sign in (-1, 0, 1)}
    return tuple(sorted(values))


# Backward-compatible alias used by older cells or interactive work.
def single_coordinate_frequency_values(config):
    return coordinate_frequency_values(config)


def scalar_frequency_radius(n_qubits, config):
    if config.encoding_base == 3:
        return (3 ** (n_qubits * config.n_upload_layers) - 1) // 2
    return sum(
        config.encoding_base ** (wire * config.n_upload_layers + layer)
        for wire in range(n_qubits)
        for layer in range(config.n_upload_layers)
    )


def nonzero_frequency_orbit(n_qubits, config):
    values = coordinate_frequency_values(config)
    zero = (0,) * n_qubits
    return [omega for omega in itertools.product(values, repeat=n_qubits) if omega != zero]


def full_nonzero_orbit_size(n_qubits, config):
    radius = scalar_frequency_radius(n_qubits, config)
    return 2 * radius


def integer_interval(start, stop):
    return list(range(int(start), int(stop) + 1))


def format_frequency_collection(values, max_listed=120):
    values = list(values)
    if max_listed is None or len(values) <= max_listed:
        return str(values)
    head_count = max_listed // 2
    tail_count = max_listed - head_count
    return f"{values[:head_count]} ... {values[-tail_count:]} (total {len(values)})"


def scalar_frequency_spectrum(n_qubits, config):
    radius = scalar_frequency_radius(n_qubits, config)
    return integer_interval(-radius, radius)


def nonzero_scalar_frequencies(n_qubits, config):
    radius = scalar_frequency_radius(n_qubits, config)
    return integer_interval(-radius, -1) + integer_interval(1, radius)


def orbit_frequency_rows(n_qubits, config):
    radius = scalar_frequency_radius(n_qubits, config)
    nonzero = nonzero_scalar_frequencies(n_qubits, config)
    return [
        {
            "orbit_frequency": "0",
            "orbit_size": 1,
            "frequencies": [0],
            "used_for_variance": False,
        },
        {
            "orbit_frequency": f"nu_{n_qubits}",
            "orbit_size": len(nonzero),
            "frequency_interval": f"[[-{radius}, -1]] U [[1, {radius}]]",
            "frequencies": nonzero,
            "used_for_variance": True,
        },
    ]


def print_frequency_orbit_definition(n_qubits, config, max_listed=120):
    radius = scalar_frequency_radius(n_qubits, config)
    spectrum = scalar_frequency_spectrum(n_qubits, config)
    rows = orbit_frequency_rows(n_qubits, config)
    variance_row = next(row for row in rows if row["used_for_variance"])

    print(f"n={n_qubits}, L={config.n_upload_layers}")
    print(f"frequency spectrum Omega = [[-{radius}, {radius}]]")
    print(f"frequency spectrum list = {format_frequency_collection(spectrum, max_listed=max_listed)}")
    print("orbit frequencies:")
    for row in rows:
        print(f"  orbit frequency {row['orbit_frequency']}: orbit size = {row['orbit_size']}")
        if "frequency_interval" in row:
            print(f"    frequencies = {row['frequency_interval']}")
            print(f"    preview = {format_frequency_collection(row['frequencies'], max_listed=max_listed)}")
        else:
            print(f"    frequencies = {format_frequency_collection(row['frequencies'], max_listed=max_listed)}")
    print(
        f"variance experiment uses orbit frequency {variance_row['orbit_frequency']} "
        f"with orbit size {variance_row['orbit_size']}"
    )
    print()


def coefficient_vector_for_orbit(coeffs, n_qubits, config):
    values = coordinate_frequency_values(config)
    coordinate_indices = np.array([freq % config.nx for freq in values], dtype=int)
    orbit_block = coeffs[np.ix_(*([coordinate_indices] * n_qubits))].reshape(-1)
    zero_position = values.index(0)
    zero_flat_index = np.ravel_multi_index(
        (zero_position,) * n_qubits,
        (len(values),) * n_qubits,
    )
    return np.delete(orbit_block, zero_flat_index)


def complex_sample_variance(samples, axis=0):
    centered = samples - np.mean(samples, axis=axis, keepdims=True)
    return np.mean(np.abs(centered) ** 2, axis=axis)


def orbit_variance_statistics(n_qubits, config, device):
    n_blocks = trainable_block_count(n_qubits, config)
    cache = make_cache(n_qubits, config, device)
    generator = torch.Generator(device="cpu")
    generator.manual_seed(config.seed + 10_000 * n_qubits)

    frequency_values = coordinate_frequency_values(config)
    orbit_size = full_nonzero_orbit_size(n_qubits, config)
    coefficient_sum = np.zeros(orbit_size, dtype=np.complex128)
    coefficient_abs2_sum = np.zeros(orbit_size, dtype=np.float64)
    aggregate_samples = np.zeros(config.n_parameter_samples, dtype=np.complex128)

    start = time.perf_counter()
    with torch.no_grad():
        for sample in range(config.n_parameter_samples):
            weights = random_weights(n_qubits, config, generator, device)
            f_grid = evaluate_function_grid(n_qubits, weights, cache, config, device)
            coeffs = coefficient_tensor(f_grid)
            coeff_vector = coefficient_vector_for_orbit(coeffs, n_qubits, config)
            coefficient_sum += coeff_vector
            coefficient_abs2_sum += np.abs(coeff_vector) ** 2
            aggregate_samples[sample] = np.sum(coeff_vector)

    coefficient_mean = coefficient_sum / config.n_parameter_samples
    coefficient_variances = coefficient_abs2_sum / config.n_parameter_samples - np.abs(coefficient_mean) ** 2
    coefficient_variances = np.maximum(coefficient_variances.real, 0.0)
    aggregate_variance = complex_sample_variance(aggregate_samples, axis=0)
    mean_coefficient_variance = float(np.mean(coefficient_variances))
    ratio = float(aggregate_variance / mean_coefficient_variance)

    return {
        "n_qubits": n_qubits,
        "n_upload_layers": config.n_upload_layers,
        "encoding_base": config.encoding_base,
        "frequency_values": frequency_values,
        "frequency_spectrum_radius": scalar_frequency_radius(n_qubits, config),
        "variance_orbit_frequency": f"nu_{n_qubits}",
        "symmetry": "zero vs nonzero frequency orbit",
        "setting": config.setting_name,
        "trainable_blocks": n_blocks,
        "orbit_size": orbit_size,
        "ratio": ratio,
        "ideal_ratio": orbit_size,
        "aggregate_variance": float(aggregate_variance),
        "mean_coefficient_variance": mean_coefficient_variance,
        "elapsed_seconds": time.perf_counter() - start,
    }


def run_variance_ratio_experiment(config=CONFIG, device=DEVICE):
    print(f"\nRunning {config.setting_name}")
    results = []
    for n_qubits in config.qubit_counts:
        stats = orbit_variance_statistics(n_qubits, config, device)
        results.append(stats)
        print(
            f"n={n_qubits:2d} | blocks={stats['trainable_blocks']:2d} | "
            f"variance orbit={stats['variance_orbit_frequency']} | "
            f"orbit size={stats['orbit_size']:7d} | "
            f"ratio={stats['ratio']:9.3f} | ratio/orbit={stats['ratio']/stats['orbit_size']:7.3f} | "
            f"time={stats['elapsed_seconds']:6.2f}s"
        )
    return results


## Frequencies Used in the Experiment

The helper below prints, for every $n$ in `CONFIG.qubit_counts`:

- the frequency spectrum $\Omega$,
- each orbit frequency and its orbit size,
- which frequencies belong to each orbit frequency,
- which orbit frequency is used for the variance experiment.

In the current experiment, the figures use orbit frequency $\nu_n$, meaning all nonzero frequencies in $\Omega$.


In [137]:
def print_observed_frequency_orbits(qubit_counts=CONFIG.qubit_counts, config=CONFIG, max_listed=120):
    for n_qubits in qubit_counts:
        print_frequency_orbit_definition(n_qubits, config, max_listed=max_listed)


print_observed_frequency_orbits()


n=1, L=2
frequency spectrum Omega = [[-4, 4]]
frequency spectrum list = [-4, -3, -2, -1, 0, 1, 2, 3, 4]
orbit frequencies:
  orbit frequency 0: orbit size = 1
    frequencies = [0]
  orbit frequency nu_1: orbit size = 8
    frequencies = [[-4, -1]] U [[1, 4]]
    preview = [-4, -3, -2, -1, 1, 2, 3, 4]
variance experiment uses orbit frequency nu_1 with orbit size 8

n=2, L=2
frequency spectrum Omega = [[-40, 40]]
frequency spectrum list = [-40, -39, -38, -37, -36, -35, -34, -33, -32, -31, -30, -29, -28, -27, -26, -25, -24, -23, -22, -21, -20, -19, -18, -17, -16, -15, -14, -13, -12, -11, -10, -9, -8, -7, -6, -5, -4, -3, -2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40]
orbit frequencies:
  orbit frequency 0: orbit size = 1
    frequencies = [0]
  orbit frequency nu_2: orbit size = 80
    frequencies = [[-40, -1]] U [[1, 40]]
    preview = [-40, -39, -38, -37, -36, -35, -34, -33

In [ ]:
results_blocks_n = run_variance_ratio_experiment(CONFIG, DEVICE)
results_blocks_1 = run_variance_ratio_experiment(BLOCK1_CONFIG, DEVICE)

# Convenience alias used only for quick interactive inspection.
results = results_blocks_n
result_sets = {
    "blocks_n": results_blocks_n,
    "blocks_1": results_blocks_1,
}


In [ ]:
def result_arrays(results):
    return {
        "n": np.array([item["n_qubits"] for item in results], dtype=float),
        "orbit": np.array([item["ideal_ratio"] for item in results], dtype=float),
        "ratio": np.array([item["ratio"] for item in results], dtype=float),
        "mean_c": np.array([item["mean_coefficient_variance"] for item in results], dtype=float),
        "var_a": np.array([item["aggregate_variance"] for item in results], dtype=float),
    }


def plot_variance_ratio(results_blocks_n, results_blocks_1=None, output_path=FIGURE_PATH):
    blocks_n = result_arrays(results_blocks_n)

    fig, ax = plt.subplots(figsize=(6.8, 4.6), constrained_layout=True)
    ax.plot(
        blocks_n["n"],
        blocks_n["orbit"],
        "--",
        color="#1E4E8C",
        linewidth=2.1,
        label=r"orbit size $|[\nu_n]|=3^{nL}-1$",
    )
    ax.plot(
        blocks_n["n"],
        blocks_n["ratio"],
        "o-",
        color="#B84A62",
        markerfacecolor="#D56A7D",
        markeredgecolor="white",
        markeredgewidth=0.8,
        linewidth=2.4,
        markersize=7.0,
        label=results_blocks_n[0].get("setting", r"StrongEntangle blocks, $B=n$"),
    )
    if results_blocks_1 is not None:
        block1 = result_arrays(results_blocks_1)
        ax.plot(
            block1["n"],
            block1["ratio"],
            "^-",
            color="#2F6F4E",
            markerfacecolor="#55A36A",
            markeredgecolor="white",
            markeredgewidth=0.8,
            linewidth=2.25,
            markersize=7.0,
            label=results_blocks_1[0].get("setting", "approx setting"),
        )

    ax.set_yscale("log", base=3)
    ax.set_xlabel("Number of qubits n")
    ax.set_ylabel(r"Variance ratio $\rho_n$")
    ax.set_title("Variance ratio")
    ax.grid(True, which="both", linewidth=0.45, alpha=0.55)
    ax.legend(loc="upper left", frameon=True, framealpha=0.95)
    ax.set_xticks(blocks_n["n"].astype(int))

    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"saved figure to {output_path}")


plot_variance_ratio(results_blocks_n, results_blocks_1)


In [ ]:
def save_results(results, output_path=RESULTS_PATH):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez(
        output_path,
        n_qubits=np.array([item["n_qubits"] for item in results], dtype=int),
        n_upload_layers=np.array([item["n_upload_layers"] for item in results], dtype=int),
        encoding_base=np.array([item["encoding_base"] for item in results], dtype=int),
        frequency_values=np.array([item["frequency_values"] for item in results], dtype=object),
        frequency_spectrum_radius=np.array([item["frequency_spectrum_radius"] for item in results], dtype=int),
        variance_orbit_frequency=np.array([item["variance_orbit_frequency"] for item in results], dtype=object),
        trainable_blocks=np.array([item["trainable_blocks"] for item in results], dtype=int),
        orbit_size=np.array([item["orbit_size"] for item in results], dtype=int),
        ratio=np.array([item["ratio"] for item in results], dtype=float),
        aggregate_variance=np.array([item["aggregate_variance"] for item in results], dtype=float),
        mean_coefficient_variance=np.array([item["mean_coefficient_variance"] for item in results], dtype=float),
        setting=np.array([item["setting"] for item in results], dtype=object),
    )
    print(f"saved results to {output_path}")


save_results(results_blocks_n, RESULTS_PATH)
save_results(results_blocks_1, RESULTS_PATH.with_name("variance_ratio_results_blocks_1.npz"))


## Raw Variance Values

This plot shows $\operatorname{Var}(a_{\nu_n})$ and $\overline{\operatorname{Var}(c_\omega)}$ for $\omega\in\Omega\setminus\{0\}$.

The code also prints the frequency spectrum $\Omega$ and confirms that the plotted variance uses orbit frequency $\nu_n$.


In [ ]:
def print_raw_variance_figure_nu(results, config=CONFIG, max_listed=120):
    n_values = [int(item["n_qubits"]) for item in results]
    print("Raw variance figure uses these orbit frequencies:")
    for n_qubits in n_values:
        print_frequency_orbit_definition(n_qubits, config, max_listed=max_listed)


def plot_raw_variances(results_blocks_n, results_blocks_1=None, output_path=None, config=CONFIG):
    if output_path is None:
        output_path = PAPER_WORK / "MSQE" / "figures" / "variance_values.png"

    print_raw_variance_figure_nu(results_blocks_n, config=config)
    blocks_n = result_arrays(results_blocks_n)

    fig, ax = plt.subplots(figsize=(6.8, 4.6), constrained_layout=True)
    line_var_a_n, = ax.plot(
        blocks_n["n"],
        blocks_n["var_a"],
        "s-",
        color="#B84A62",
        markerfacecolor="#D56A7D",
        markeredgecolor="white",
        markeredgewidth=0.8,
        linewidth=2.35,
        markersize=6.8,
        label=r"$\operatorname{Var}(a_{\nu_n})$, $B=n$",
    )
    visible_values = [blocks_n["var_a"], blocks_n["mean_c"]]

    line_var_a_1 = None
    line_mean_c_1 = None
    if results_blocks_1 is not None:
        block1 = result_arrays(results_blocks_1)
        line_var_a_1, = ax.plot(
            block1["n"],
            block1["var_a"],
            "s--",
            color="#B84A62",
            markerfacecolor="white",
            markeredgecolor="#B84A62",
            markeredgewidth=1.1,
            linewidth=2.0,
            markersize=6.6,
            alpha=0.9,
            label=r"$\operatorname{Var}(a_{\nu_n})$, $B=1$",
        )
        visible_values.append(block1["var_a"])

    line_mean_c_n, = ax.plot(
        blocks_n["n"],
        blocks_n["mean_c"],
        "o-",
        color="#1E4E8C",
        markerfacecolor="#3B78C2",
        markeredgecolor="white",
        markeredgewidth=0.8,
        linewidth=2.35,
        markersize=6.8,
        label=r"$\mathbb{E}_{\omega}\operatorname{Var}(c_{\omega})$, $B=n$",
    )

    if results_blocks_1 is not None:
        line_mean_c_1, = ax.plot(
            block1["n"],
            block1["mean_c"],
            "o--",
            color="#1E4E8C",
            markerfacecolor="white",
            markeredgecolor="#1E4E8C",
            markeredgewidth=1.1,
            linewidth=2.0,
            markersize=6.6,
            alpha=0.9,
            label=r"$\mathbb{E}_{\omega}\operatorname{Var}(c_{\omega})$, $B=1$",
        )
        visible_values.append(block1["mean_c"])

    positive_values = np.concatenate(visible_values)
    positive_values = positive_values[np.isfinite(positive_values) & (positive_values > 0)]
    if positive_values.size:
        y_min = 10 ** np.floor(np.log10(np.min(positive_values)) - 0.25)
        y_max = 10 ** np.ceil(np.log10(np.max(positive_values)) + 0.25)
        ax.set_ylim(y_min, y_max)

    ax.set_yscale("log")
    ax.set_xlabel("Number of qubits n")
    ax.set_ylabel("Coefficient variance")
    ax.set_title("Variance values")
    ax.grid(True, which="both", linewidth=0.45, alpha=0.55)

    handles = [line_var_a_n]
    if line_var_a_1 is not None:
        handles.append(line_var_a_1)
    handles.append(line_mean_c_n)
    if line_mean_c_1 is not None:
        handles.append(line_mean_c_1)
    ax.legend(handles=handles, loc="best", frameon=True, framealpha=0.95)
    ax.set_xticks(blocks_n["n"].astype(int))

    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"saved figure to {output_path}")


plot_raw_variances(results_blocks_n, results_blocks_1, config=CONFIG)


## Constructed Symmetry Overcoming Vanishing Expressivity

This optional cell is a coefficient-level toy demo, separate from the circuit figures above.

For each $n$, it chooses one orbit with size
$$
R_n=\left\lceil \frac{b^n}{c(n)}\right\rceil.
$$
It samples independent coefficients with variance $b^{-n}$. The aggregate coefficient then has variance
$$
\operatorname{Var}(a_{\nu_n})=R_n b^{-n}.
$$
This demonstrates the corollary condition directly, without running a quantum circuit.


In [ ]:

def plot_constructed_overcome_demo(
    n_values=None,
    b=3.0,
    c_power=2.0,
    n_mc_samples=4096,
    seed=5,
    output_path=None,
):
    """Coefficient-level construction of the corollary condition.

    Symmetry: C_R acts transitively on an orbit of size R.
    Coefficients: independent complex Gaussians with Var(c_omega)=b^{-n}.
    We sample a_nu directly because the sum of R independent complex Gaussians
    is again complex Gaussian with variance R * b^{-n}.
    """
    if n_values is None:
        n_values = np.arange(1, 11, dtype=int)
    else:
        n_values = np.array(n_values, dtype=int)
    if output_path is None:
        output_path = PAPER_WORK / "MSQE" / "figures" / "constructed_overcome_vanishing.png"

    rng = np.random.default_rng(seed)
    c_values = np.maximum(n_values.astype(float) ** c_power, 1.0)
    single_var_exact = b ** (-n_values.astype(float))
    orbit_sizes = np.ceil((b ** n_values.astype(float)) / c_values).astype(object)
    aggregate_var_exact = np.array(
        [float(orbit_size) * single_var for orbit_size, single_var in zip(orbit_sizes, single_var_exact)],
        dtype=float,
    )
    target_lower_bound = 1.0 / c_values

    single_var_empirical = []
    aggregate_var_empirical = []
    for single_var, aggregate_var in zip(single_var_exact, aggregate_var_exact):
        c_samples = (
            rng.normal(size=n_mc_samples) + 1j * rng.normal(size=n_mc_samples)
        ) * np.sqrt(single_var / 2.0)
        a_samples = (
            rng.normal(size=n_mc_samples) + 1j * rng.normal(size=n_mc_samples)
        ) * np.sqrt(aggregate_var / 2.0)
        single_var_empirical.append(complex_sample_variance(c_samples, axis=0))
        aggregate_var_empirical.append(complex_sample_variance(a_samples, axis=0))

    single_var_empirical = np.array(single_var_empirical, dtype=float)
    aggregate_var_empirical = np.array(aggregate_var_empirical, dtype=float)

    print("n | orbit size | Var(c_omega) exact | Var(a_nu) exact | 1/c(n)")
    for n, orbit_size, vc, va, target in zip(
        n_values, orbit_sizes, single_var_exact, aggregate_var_exact, target_lower_bound
    ):
        print(f"{n:2d} | {int(orbit_size):10d} | {vc:17.4e} | {va:15.4e} | {target:8.4e}")

    fig, ax = plt.subplots(figsize=(6.8, 4.6), constrained_layout=True)
    ax.plot(
        n_values,
        single_var_empirical,
        "o-",
        color="#1E4E8C",
        markerfacecolor="#3B78C2",
        markeredgecolor="white",
        markeredgewidth=0.8,
        linewidth=2.4,
        markersize=7.0,
        label=r"sampled $\operatorname{Var}(c_\omega)$",
    )
    ax.plot(
        n_values,
        aggregate_var_empirical,
        "s-",
        color="#B84A62",
        markerfacecolor="#D56A7D",
        markeredgecolor="white",
        markeredgewidth=0.8,
        linewidth=2.4,
        markersize=7.0,
        label=r"sampled $\operatorname{Var}(a_\nu)$",
    )
    ax.plot(
        n_values,
        target_lower_bound,
        "--",
        color="#2F6F4E",
        linewidth=2.1,
        label=r"target $1/c(n)$",
    )

    ax.set_yscale("log")
    ax.set_xlabel("Number of qubits n")
    ax.set_ylabel("Coefficient variance")
    ax.set_title("Constructed variance values")
    ax.grid(True, which="both", linewidth=0.45, alpha=0.55)
    ax.legend(loc="best", frameon=True, framealpha=0.95)
    ax.set_xticks(n_values.astype(int))

    positive_values = np.concatenate([single_var_empirical, aggregate_var_empirical, target_lower_bound])
    positive_values = positive_values[np.isfinite(positive_values) & (positive_values > 0)]
    if positive_values.size:
        y_min = 10 ** np.floor(np.log10(np.min(positive_values)) - 0.25)
        y_max = 10 ** np.ceil(np.log10(np.max(positive_values)) + 0.25)
        ax.set_ylim(y_min, y_max)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"saved figure to {output_path}")
    return {
        "n_values": n_values,
        "orbit_sizes": np.array([int(orbit_size) for orbit_size in orbit_sizes], dtype=object),
        "single_var_exact": single_var_exact,
        "aggregate_var_exact": aggregate_var_exact,
        "single_var_empirical": single_var_empirical,
        "aggregate_var_empirical": aggregate_var_empirical,
        "target_lower_bound": target_lower_bound,
    }


constructed_results = plot_constructed_overcome_demo(b=3.0, c_power=2.0)


## Reading the Diagnostics

- `ratio` is $\operatorname{Var}(a_{\nu_n})/\overline{\operatorname{Var}(c_\omega)}_{\omega\in\Omega\setminus\{0\}}$.
- `ratio/orbit` close to one means the empirical ratio follows the ideal 2-design scaling.
- The two circuit settings compared here are StrongEntangle-style trainable layers with `B=n` blocks and `B=1` block.
- Increase `n_parameter_samples` for a smoother curve; runtime is roughly linear in this value.
